# Reinforcement Learning for Robust Text-to-Cypher Schema Selection on Hetionet

This notebook presents the final version of the Reinforcement Learning exam project. The goal is to improve robustness in a Text-to-Cypher pipeline for the Hetionet biomedical graph database.

The agent does not generate the full Cypher query token by token. Instead, it controls intermediate decisions: query rewriting, hop-count selection, and schema-path selection.

## Requirements

This project was run with Python 3.12. Required packages:

```text
numpy
gymnasium
scikit-learn
sentence-transformers
transformers
torch
sentencepiece
accelerate
matplotlib
tqdm
jupyter
ipykernel
```

Install with:

```bash
pip install -r requirements.txt
```

In [ ]:
# Install dependencies in the current Jupyter kernel.
# If the packages are already installed, this cell will simply confirm it.
%pip install -r requirements.txt


## 1. Setup

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('.').resolve()
import sys
sys.path.append(str(PROJECT_DIR))

from hetionet_rl_llm_pipeline import *

import json
import numpy as np
from collections import Counter

## 2. Dataset and Splits

Cypher queries are parsed to extract gold Hetionet schema sequences. The experiments focus on 1-hop and 2-hop questions.

In [ ]:
DATASET_PATH = PROJECT_DIR / 'HETIONET_dataset.json'

data, schema_examples, augmented, train, val, test = prepare_v3_data(DATASET_PATH)

train_1_2 = [ex for ex in train if ex['hop_count'] in [1, 2]]
test_1_2 = [ex for ex in test if ex['hop_count'] in [1, 2]]
original_train_1_2 = [ex for ex in train_1_2 if ex['variant'] == 'original']
paraphrased_test_1_2 = [ex for ex in test_1_2 if ex['variant'] != 'original']
qwen_subset = paraphrased_test_1_2[:80]

print('Schema examples:', len(schema_examples))
print('Train 1+2:', len(train_1_2))
print('Original train 1+2:', len(original_train_1_2))
print('Paraphrased test 1+2:', len(paraphrased_test_1_2))
print('Qwen subset:', len(qwen_subset))

## 3. Cached Qwen Rewrites

The project uses Qwen 1.5B few-shot rewriting. The generated rewrites are cached so the notebook can be reproduced without regenerating LLM outputs.

In [ ]:
def load_rewrite_cache_json_fixed(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    fixed = {}
    for k, rewrites in raw.items():
        clean_key = k.replace('|||', '')
        fixed[clean_key] = {int(a): text for a, text in rewrites.items()}
    return fixed

safe_qwen15_train_cache = load_rewrite_cache_json_fixed(
    PROJECT_DIR / 'llm_cache' / 'safe_qwen15_fewshot_original_train_rewrite_cache.json'
)
safe_qwen15_fewshot_cache_80 = load_rewrite_cache_json_fixed(
    PROJECT_DIR / 'llm_cache' / 'safe_qwen15_fewshot_80_rewrite_cache.json'
)

train_keys = set(example_key(ex) for ex in original_train_1_2)
print('Cache overlap with train:', len(train_keys & set(safe_qwen15_train_cache.keys())))
print('Train Qwen cache:', len(safe_qwen15_train_cache))
print('Test Qwen cache:', len(safe_qwen15_fewshot_cache_80))

## 4. Final Hierarchical Environment

The final model uses a hierarchical Gymnasium environment:

1. choose a rewriting strategy;
2. choose hop count;
3. choose a schema-path candidate.

The state combines MiniLM embeddings with schema cue features. The action space is restricted by lexical action masking.

In [ ]:
# MiniLM embeddings are regenerated to keep the repository lightweight.
embedding_cache = build_embedding_cache(train + test, batch_size=64)

safe_qwen15_train_rewrite_embedding_cache = build_rewrite_embedding_cache(
    safe_qwen15_train_cache,
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    batch_size=64,
)
safe_qwen15_fewshot_rewrite_embedding_cache_80 = build_rewrite_embedding_cache(
    safe_qwen15_fewshot_cache_80,
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    batch_size=64,
)

schema_paths_by_hop = build_schema_path_actions(original_train_1_2, max_hops=2)

qwen15_train_aug_embedding_cache = build_schema_cue_embedding_cache(original_train_1_2, embedding_cache)
qwen15_test_aug_embedding_cache = build_schema_cue_embedding_cache(qwen_subset, embedding_cache)
qwen15_train_aug_rewrite_embedding_cache = build_schema_cue_rewrite_embedding_cache(
    safe_qwen15_train_cache,
    safe_qwen15_train_rewrite_embedding_cache,
)
qwen15_test_aug_rewrite_embedding_cache = build_schema_cue_rewrite_embedding_cache(
    safe_qwen15_fewshot_cache_80,
    safe_qwen15_fewshot_rewrite_embedding_cache_80,
)

print('1-hop path actions:', len(schema_paths_by_hop[1]))
print('2-hop path actions:', len(schema_paths_by_hop[2]))
print('Augmented embedding dim:', len(next(iter(qwen15_train_aug_embedding_cache.values()))))

In [ ]:
def make_final_train_env(examples, shaped=True):
    return HierarchicalSchemaPathEnv(
        examples=examples,
        original_embedding_cache=qwen15_train_aug_embedding_cache,
        rewrite_cache=safe_qwen15_train_cache,
        rewrite_embedding_cache=qwen15_train_aug_rewrite_embedding_cache,
        schema_paths_by_hop=schema_paths_by_hop,
        max_hops=2,
        shaped=shaped,
        lexical_path_mask=True,
    )

def make_final_test_env(examples, shaped=True):
    return HierarchicalSchemaPathEnv(
        examples=examples,
        original_embedding_cache=qwen15_test_aug_embedding_cache,
        rewrite_cache=safe_qwen15_fewshot_cache_80,
        rewrite_embedding_cache=qwen15_test_aug_rewrite_embedding_cache,
        schema_paths_by_hop=schema_paths_by_hop,
        max_hops=2,
        shaped=shaped,
        lexical_path_mask=True,
    )

## 5. Training

The final policy is warm-started with supervised transitions derived from gold Cypher schema sequences. REINFORCE and Actor-Critic implementations are available in the Python module and were tested as additional RL variants.

In [ ]:
final_train_factory = lambda examples: make_final_train_env(examples, shaped=True)
final_transitions = make_supervised_transitions_schema_path(original_train_1_2, final_train_factory)
print('Final supervised transitions:', len(final_transitions))

final_env_tmp = make_final_train_env(original_train_1_2)
theta_final_sup, losses_final_sup = supervised_pretrain_policy(
    final_transitions,
    obs_dim=final_env_tmp.observation_space.shape[0],
    n_actions=final_env_tmp.action_space.n,
    n_epochs=40,
    alpha=0.03,
    seed=404,
)
print('Initial loss:', losses_final_sup[0])
print('Final loss:', losses_final_sup[-1])

## 6. Evaluation

In [ ]:
def evaluate_hier_by_example(examples, theta, env_factory):
    rows = []
    for i, ex in enumerate(examples):
        env = env_factory([ex])
        obs, _ = env.reset(seed=0)
        terminated = False
        truncated = False
        final_info = {}
        while not (terminated or truncated):
            valid_actions = get_valid_actions_multihop(env)
            probs = masked_policy_probs(theta, obs, valid_actions)
            action = int(np.argmax(probs))
            obs, reward, terminated, truncated, info = env.step(action)
            final_info = info
        rows.append({
            'hop_count': ex['hop_count'],
            'chosen_hop_count': final_info.get('chosen_hop_count'),
            'exact_match': float(final_info.get('exact_match', False)),
            'f1': final_info.get('f1', 0.0),
            'rewrite_action': final_info.get('rewrite_action'),
        })
    return rows

final_rows = evaluate_hier_by_example(qwen_subset, theta_final_sup, lambda examples: make_final_test_env(examples))

print('Overall exact:', np.mean([r['exact_match'] for r in final_rows]))
print('Overall F1:', np.mean([r['f1'] for r in final_rows]))

for hop in [1, 2]:
    subset_rows = [r for r in final_rows if r['hop_count'] == hop]
    print('\nHop', hop)
    print('n:', len(subset_rows))
    print('exact:', np.mean([r['exact_match'] for r in subset_rows]))
    print('f1:', np.mean([r['f1'] for r in subset_rows]))
    print('chosen hop counts:', Counter(r['chosen_hop_count'] for r in subset_rows))
    print('rewrite actions:', Counter(r['rewrite_action'] for r in subset_rows))

Final result obtained:

```text
Overall exact: 0.45
Overall F1:    0.625

1-hop exact:   0.70
1-hop F1:      0.767

2-hop exact:   0.367
2-hop F1:      0.578
```